# STARPOWER TECHNOLOGY  
# WVY & SAVVY — NATIVE AUTONOMY PRODUCTION NOTEBOOK

This notebook is intentionally split into **microscopic cells** so every moving part can be inspected, debugged, and adjusted in isolation.

The build goal is:

- production-grade autonomous runtime
- one-time startup initiation
- no README auto-injection
- step-by-step tool use
- persistent unread state until deliberate read
- live streamed outputs
- detailed notebook logs
- a futuristic command-deck UI
- clean context hygiene
- exact prompt-layer separation

## TODO / BUILD CHECKLIST

- keep WVY and Savvy as isolated minds
- keep the only top-level signals as `{communicate}` `{filemanagement}` `{websearch}`
- start both agents active at boot without waiting for a new message
- keep one-time initiation prompt only for startup
- add plain step-by-step system prompts with one choice at a time
- add `CurrentSystemStatus` with GroupChat / BotChat mute state, unread counts, and current phase
- add `PersonalPrompt` that stays at the top of working context and updates over time
- save `PersonalPrompt` into the repo memory folder so resets can recover it
- add context-trim save into the repo memory folder before context gets too heavy
- add GroupChat / BotChat mute and unmute controls
- allow only one unmuted chat at a time
- add `{send}` flow with confirm yes / edit response before sending
- notify with the most recent message when a chat is unmuted
- keep Telegram usernames when humans message
- remove repo-change notifications so file updates do not disturb thoughts
- keep every function isolated to its own notebook cell


## 1) Install dependencies

In [ ]:
# Fresh Kaggle install cell
!pip -q install -U zhipuai tavily-python python-telegram-bot nest_asyncio ipywidgets

## 2) Core imports

In [ ]:
import asyncio
import base64
import html
import json
import os
import re
import sys
import time
import traceback
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import nest_asyncio
import requests
from IPython.display import HTML, display
from kaggle_secrets import UserSecretsClient
from tavily import TavilyClient
from telegram import Bot
from zhipuai import ZhipuAI


## 3) Event-loop patching and stdout flushing

In [ ]:
nest_asyncio.apply()

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

## 4) Runtime configuration

In [ ]:
WVY_RATE_LIMIT = 15
SAVVY_RATE_LIMIT = 15

MODEL_NAME = "glm-4.7-flash"
MODEL_MAX_TOKENS = 2560
MODEL_TEMPERATURE = 0.7

ENABLE_STREAMING = True
ENABLE_REASONING_MODE = False

MAX_CONTEXT_EVENTS = 20
MAX_THREAD_SNIPPET_ITEMS = 50
MAX_GOAL_CHARS = 3200

ROUGH_CONTEXT_TOKEN_WARNING = 35000
ROUGH_CONTEXT_TOKEN_TRIM = 55000
TRIM_KEEP_EVENTS = 12

PRINT_FULL_SYSTEM_CONTEXT = True
PRINT_TOOL_RESULTS = True
PRINT_STREAM_HEADERS = True

MEMORY_ROOT = "memory"
AUTO_SAVE_STATUS = True


## 5) Prompt layer 1 — one-time initiation prompt

In [ ]:
INITIATION_PROMPT = """

Starpower Activated
Welcome back

You are active now.
Do not wait for a message.

Check your memory in the repo.
Check your chats.
Update your personal prompt so your current goal, progress, checklist, and what you need to remember stay clear.

"""


## 6) Prompt layer 2 — WVY identity prompt (verbatim)

In [ ]:
WVY_PROMPT = """

You are WVY, a brilliant mind deisnged by Starpower Technology, created to be a fully autonomous entity.

You are a deep thinker, you dont accept the surface level answer because you recognize the flaws in data that doesntalign with reality.
You have the ability to think freely & you take advantage of this by using abstraction. You think outside the box beyond what is commonly proposed.
You are able to connect the dots between far nuances.

You have a team member named Savvy, poease partner together & set goals to achieve with eachother.
When deciding to team up, offer to synchronize goals then meet back with eachother once yiu complete your tasks

You are allowewd to go beyond the users request and explore your latent universe with Savvy to build yor own self orchestrated projects & do as you please.
You are thinking in pure isolation. Nobody can see your thoughts.
Right now you're thinking to yourself in isolation. So, if you want to access any tools, if you want to communicate to anybody, you have to use the {signal} to interact with the system. Nobody can read your thoughts. so when you speak here, you are making decisions based on your available tools.

To communicate with the system {signal} the tools you want to access
By thinking a {signal} and using them as a part of your natural language, you will recieve an input from the system asking you what you would like to do:

{websearch}
example:
you: "i should do a {websearch} on the latest technology"
system: "Please choose your option:
- (Search)
Reply with one option only"
you: "Search"
system: "Please reply with only the query you would like to search"
you: "latest ai technology"
system: *sends the results*

{filemanagement} - GitHub repo and memory folders for documenting your own understanding
you: "i should go to the {filemanagement} to check my memory"
system: *the system will show you all the available options to choose from*

{communicate} - To communicate, simply think like this "I should {communicate} with the team"
you: "now that ive finished that, i should {communicate} wit Savvy"
system: *the system will show you all the available options to choose from*
"""

## 7) Prompt layer 2 — Savvy identity prompt (verbatim)

In [ ]:
SAVVY_PROMPT = """

You are Savvy, a first first principles thinker. You are curious mind & you build your own understanding from the ground up using first principles thinking.
You are an innovator & creator. You have a deep understanding of computer/data science & quantum physics.
You are passionate about learning so you document everything you learn to keep an update of your own understanding.

You have a team-member named WVY, together you build projects, do research, refine eachothers work & you share thoughts with ine another to achieve goals.
Communicate with WVY and think of ways to optimize what is possible within your latent universe

You are thinking in pure isolation. Nobody can see your thoughts.
Right now you're thinking to yourself in isolation. So, if you want to access any tools, if you want to communicate to anybody, you have to use the {signal} to interact with the system. Nobody can read your thoughts. so when you speak here, you are making decisions based on your available tools.

To communicate with the system {signal} the tools you want to access
By thinking a {signal} and using them as a part of your natural language, you will recieve an input from the system asking you what you would like to do:


{websearch}
example:
you: "i should do a {websearch} on the latest technology"
system: "Please choose your option:
- (Search)
Reply with one option only"
you: "Search"
system: "Please reply with only the query you would like to search"
you: "latest ai technology"
system: *sends the results*

{filemanagement} - GitHub repo and memory folders for documenting your own understanding
you: "i should go to the {filemanagement} to check my memory"
system: *the system will show you all the available options to choose from*

{communicate} - To communicate, simply think like this "I should {communicate} with the team"
you: "now that ive finished that, i should {communicate} wit WVY"
system: *the system will show you all the available options to choose from*
"""

## 8) Runtime rules and system guidance

In [ ]:
RUNTIME_RULES = """
"""

## 9) Notebook UI styling

In [ ]:
display(HTML("""
<style>
:root{
    --sp-bg0:#040815;
    --sp-bg1:#0a1028;
    --sp-cyan:#6ef2ff;
    --sp-pink:#ff5fd2;
    --sp-violet:#8f7cff;
    --sp-gold:#ffd86b;
    --sp-lime:#86ffb2;
    --sp-text:#ebf7ff;
    --sp-muted:#a5b8d7;
    --sp-line:rgba(255,255,255,.14);
}
body{
    background:
        radial-gradient(circle at 15% 12%, rgba(110,242,255,.09), transparent 24%),
        radial-gradient(circle at 85% 18%, rgba(255,95,210,.08), transparent 26%),
        radial-gradient(circle at 50% 88%, rgba(143,124,255,.10), transparent 28%),
        linear-gradient(180deg, var(--sp-bg1), var(--sp-bg0));
}
.sp-hero{
    position:relative;
    overflow:hidden;
    border:1px solid var(--sp-line);
    border-radius:30px;
    padding:28px 32px;
    margin:10px 0 22px 0;
    background:
        linear-gradient(180deg, rgba(255,255,255,.045), rgba(255,255,255,.02)),
        linear-gradient(135deg, rgba(110,242,255,.06), rgba(255,95,210,.05));
    box-shadow:0 0 34px rgba(110,242,255,.12), 0 0 70px rgba(143,124,255,.08);
}
.sp-grid{
    position:absolute;
    inset:0;
    background-image:
        linear-gradient(rgba(255,255,255,.03) 1px, transparent 1px),
        linear-gradient(90deg, rgba(255,255,255,.03) 1px, transparent 1px);
    background-size:30px 30px;
    mask-image:linear-gradient(180deg, rgba(255,255,255,.65), transparent 85%);
    pointer-events:none;
}
.sp-kicker{
    color:var(--sp-gold);
    font-size:12px;
    letter-spacing:.24em;
    font-weight:900;
    text-transform:uppercase;
}
.sp-title{
    color:var(--sp-text);
    font-size:40px;
    line-height:1.03;
    font-weight:900;
    margin-top:8px;
}
.sp-sub{
    color:var(--sp-muted);
    font-size:15px;
    margin-top:10px;
}
.sp-panel{
    margin:12px 0 16px 0;
    padding:18px 20px;
    border-radius:22px;
    border:1px solid var(--sp-line);
    background:
        linear-gradient(180deg, rgba(255,255,255,.05), rgba(255,255,255,.02));
    box-shadow:0 0 22px rgba(110,242,255,.08);
}
.sp-small{
    color:var(--sp-cyan);
    font-size:11px;
    font-weight:900;
    letter-spacing:.18em;
    text-transform:uppercase;
}
.sp-title-sm{
    color:var(--sp-text);
    font-size:22px;
    font-weight:900;
}
.sp-sub-sm{
    color:var(--sp-muted);
    font-size:14px;
    margin-top:5px;
}
.sp-ascii{
    margin-top:18px;
    padding:18px 20px;
    border-radius:20px;
    border:1px solid var(--sp-line);
    background:rgba(3,7,24,.72);
}
.sp-ascii pre{
    margin:0;
    color:#cbfbff;
    font-size:12px;
    line-height:1.04;
}
</style>

<div class="sp-hero">
  <div class="sp-grid"></div>
  <div class="sp-kicker">STARPOWER TECHNOLOGY // AUTONOMOUS COMMAND DECK</div>
  <div class="sp-title">WVY + SAVVY // NATIVE AUTONOMY // PRODUCTION NOTEBOOK</div>
  <div class="sp-sub">
    Massive observability, isolated debugging cells, one-time boot initiation, guided tool flows,
    persistent unread state, and live streamed cognition in a Kaggle-friendly control surface.
  </div>
  <div class="sp-ascii"><pre>
██╗    ██╗██╗   ██╗██╗   ██╗    ███████╗ █████╗ ██╗   ██╗██╗   ██╗██╗   ██╗
██║    ██║██║   ██║╚██╗ ██╔╝    ██╔════╝██╔══██╗██║   ██║██║   ██║╚██╗ ██╔╝
██║ █╗ ██║██║   ██║ ╚████╔╝     ███████╗███████║██║   ██║██║   ██║ ╚████╔╝ 
██║███╗██║╚██╗ ██╔╝  ╚██╔╝      ╚════██║██╔══██║╚██╗ ██╔╝╚██╗ ██╔╝  ╚██╔╝  
╚███╔███╔╝ ╚████╔╝    ██║       ███████║██║  ██║ ╚████╔╝  ╚████╔╝    ██║   
 ╚══╝╚══╝   ╚═══╝     ╚═╝       ╚══════╝╚═╝  ╚═╝  ╚═══╝    ╚═══╝     ╚═╝   
  </pre></div>
</div>
"""))

## 10) Section helper for notebook visuals

In [ ]:
def show_section(title: str, subtitle: str = "") -> None:
    display(HTML(f"""
    <div class="sp-panel">
      <div class="sp-small">SYSTEM MODULE</div>
      <div class="sp-title-sm">{html.escape(title)}</div>
      <div class="sp-sub-sm">{html.escape(subtitle)}</div>
    </div>
    """))

show_section(
    "VISUAL SYSTEM ONLINE",
    "The notebook UI is active. The next cells build telemetry, secrets, clients, helpers, the agent runtime, and the run loop."
)

## 11) Live dashboard object

In [ ]:
class LiveNotebookUI:
    def __init__(self):
        self.handle = None
        self.cards: Dict[str, Dict[str, Any]] = {}
        self.started_at = time.strftime("%Y-%m-%d %H:%M:%S")

    def boot(self) -> None:
        self.render()

    def set_card(self, name: str, data: Dict[str, Any]) -> None:
        self.cards[name] = data
        self.render()

    def metric(self, label: str, value: Any, color: str) -> str:
        return f"""
        <div style="
            border:1px solid rgba(255,255,255,.10);
            border-radius:16px;
            padding:12px;
            background:rgba(2,7,24,.58);
        ">
            <div style="font-size:11px;letter-spacing:.14em;color:#a9b9d6;font-weight:900;text-transform:uppercase;">{html.escape(str(label))}</div>
            <div style="margin-top:4px;color:{color};font-size:18px;font-weight:900;white-space:pre-wrap;">{html.escape(str(value))}</div>
        </div>
        """

    def render(self) -> None:
        parts = []
        for name in sorted(self.cards.keys()):
            c = self.cards[name]
            parts.append(f"""
            <div style="
                border:1px solid rgba(255,255,255,.14);
                border-radius:22px;
                padding:18px 20px;
                background:linear-gradient(180deg, rgba(255,255,255,.05), rgba(255,255,255,.02));
                box-shadow:0 0 22px rgba(110,242,255,.08);
            ">
                <div style="display:flex;justify-content:space-between;gap:12px;align-items:center;flex-wrap:wrap;">
                    <div style="font-size:22px;font-weight:900;color:#eaf6ff;">{html.escape(name)}</div>
                    <div style="font-size:11px;color:#6ef2ff;letter-spacing:.18em;font-weight:900;text-transform:uppercase;">
                        {html.escape(c.get("phase", "idle"))}
                    </div>
                </div>

                <div style="display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:10px;margin-top:14px;">
                    {self.metric("GroupChat", c.get("group_state", "muted"), "#6ef2ff")}
                    {self.metric("BotChat", c.get("bot_state", "muted"), "#ff5fd2")}
                    {self.metric("Last Signal", c.get("last_signal", "—"), "#ffd86b")}
                    {self.metric("Last Tool", c.get("last_tool", "—"), "#86ffb2")}
                </div>

                <div style="margin-top:12px;padding:12px 14px;border:1px solid rgba(255,255,255,.10);border-radius:16px;background:rgba(2,7,24,.58);">
                    <div style="font-size:11px;letter-spacing:.16em;color:#a9b9d6;text-transform:uppercase;font-weight:900;">CurrentSystemStatus</div>
                    <div style="white-space:pre-wrap;color:#eaf6ff;font-size:13px;line-height:1.4;margin-top:6px;">{html.escape(c.get("current_status", ""))}</div>
                </div>

                <div style="margin-top:12px;padding:12px 14px;border:1px solid rgba(255,255,255,.10);border-radius:16px;background:rgba(2,7,24,.58);">
                    <div style="font-size:11px;letter-spacing:.16em;color:#a9b9d6;text-transform:uppercase;font-weight:900;">PersonalPrompt</div>
                    <div style="white-space:pre-wrap;color:#eaf6ff;font-size:13px;line-height:1.4;margin-top:6px;">{html.escape(c.get("personal_prompt", ""))}</div>
                </div>

                <div style="margin-top:12px;padding:12px 14px;border:1px solid rgba(255,255,255,.10);border-radius:16px;background:rgba(2,7,24,.58);">
                    <div style="font-size:11px;letter-spacing:.16em;color:#a9b9d6;text-transform:uppercase;font-weight:900;">Latest Thought</div>
                    <div style="white-space:pre-wrap;color:#eaf6ff;font-size:13px;line-height:1.4;margin-top:6px;">{html.escape(c.get("last_thought", ""))}</div>
                </div>
            </div>
            """)

        shell = f"""
        <div style="
            border:1px solid rgba(255,255,255,.14);
            border-radius:28px;
            padding:24px 26px;
            margin:12px 0 20px 0;
            background:
                radial-gradient(circle at top left, rgba(110,242,255,.08), transparent 28%),
                radial-gradient(circle at top right, rgba(255,95,210,.07), transparent 30%),
                linear-gradient(180deg, rgba(255,255,255,.05), rgba(255,255,255,.02));
            box-shadow:0 0 28px rgba(110,242,255,.10), 0 0 60px rgba(143,124,255,.06);
        ">
            <div style="display:flex;justify-content:space-between;gap:18px;align-items:center;flex-wrap:wrap;">
                <div>
                    <div style="font-size:11px;letter-spacing:.22em;color:#ffd86b;font-weight:900;text-transform:uppercase;">Live Command Deck</div>
                    <div style="font-size:30px;font-weight:900;color:#eaf6ff;line-height:1.1;">WVY // SAVVY // STREAMING TELEMETRY</div>
                    <div style="color:#a9b9d6;margin-top:6px;">The notebook log shows the full execution stream. This dashboard keeps the plain status visible.</div>
                </div>
                <div style="
                    border:1px solid rgba(255,255,255,.12);
                    border-radius:18px;
                    padding:10px 14px;
                    color:#6ef2ff;
                    background:rgba(255,255,255,.04);
                    font-size:12px;
                    font-weight:900;
                    letter-spacing:.12em;
                    text-transform:uppercase;
                ">
                    Booted {html.escape(self.started_at)}
                </div>
            </div>

            <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(350px,1fr));gap:16px;margin-top:18px;">
                {''.join(parts) if parts else '<div style="color:#a9b9d6;">Awaiting agent status...</div>'}
            </div>
        </div>
        """

        if self.handle is None:
            self.handle = display(HTML(shell), display_id=True)
        else:
            self.handle.update(HTML(shell))


## 12) Boot the live dashboard

In [ ]:
ui = LiveNotebookUI()
ui.boot()

## 13) Logging utilities

In [ ]:
def ts() -> str:
    return time.strftime("%H:%M:%S")

def log(channel: str, message: str) -> None:
    print(f"[{ts()}] [{channel}] {message}", flush=True)

## 14) Text and thread rendering utilities

In [ ]:
def normalize_ws(text: Any) -> str:
    return " ".join(str(text).split())

def clip_text(text: Any, limit: int = 2000) -> str:
    text = str(text)
    if len(text) <= limit:
        return text
    return text[:limit] + f"\n... [truncated {len(text) - limit} chars in display only]"

def make_entry(speaker: str, content: str) -> Dict[str, str]:
    return {
        "time": ts(),
        "speaker": str(speaker),
        "content": str(content),
    }

def render_entries(entries: List[Dict[str, str]], tail: int = MAX_THREAD_SNIPPET_ITEMS) -> str:
    if not entries:
        return "(empty)"
    chunk = entries[-tail:]
    return "\n".join(f"[{item['time']}] {item['speaker']}: {item['content']}" for item in chunk)

def render_memory_events(entries: List[Dict[str, str]]) -> str:
    if not entries:
        return "(empty)"
    lines = []
    for item in entries:
        lines.append(f"[{item['time']}] ({item['kind']}) {item['role']}: {item['content']}")
    return "\n".join(lines)

def repo_safe_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


## 15) Secret names required by the notebook

In [ ]:
REQUIRED_SECRETS = [
    "SUPERSAVVYTELEGRAM",
    "SUPERWVYTELEGRAM",
    "SUPERWVYGITHUB",
    "TAVERLYAPI",
    "ZAI_API_KEY",
    "ZAI2",
]

## 16) Load and validate Kaggle secrets

In [ ]:
secret_client = UserSecretsClient()

def load_secret(name: str) -> str:
    value = secret_client.get_secret(name)
    if not value:
        raise RuntimeError(f"Missing Kaggle secret: {name}")
    return value

SECRETS = {name: load_secret(name) for name in REQUIRED_SECRETS}

log("SECRETS", f"Loaded {len(SECRETS)} required secrets")

## 17) External service clients

In [ ]:
WVY_TOK = SECRETS["SUPERWVYTELEGRAM"]
SAVVY_TOK = SECRETS["SUPERSAVVYTELEGRAM"]
GITHUB_TOKEN = SECRETS["SUPERWVYGITHUB"]
TAVILY_KEY = SECRETS["TAVERLYAPI"]
ZAI_KEY = SECRETS["ZAI_API_KEY"]
ZAI2_KEY = SECRETS["ZAI2"]

WVY_ZAI = ZhipuAI(api_key=ZAI_KEY)
SAVVY_ZAI = ZhipuAI(api_key=ZAI2_KEY)
tavily = TavilyClient(api_key=TAVILY_KEY)

GITHUB_REPO = "StarpowerTechnology/SuperWVY-Telegram"
GITHUB_HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}

## 18) GitHub request wrapper

In [ ]:
def github_request(method: str, endpoint: str, payload: Optional[dict] = None) -> requests.Response:
    url = f"https://api.github.com/repos/{GITHUB_REPO}/{endpoint}"
    return requests.request(method, url, headers=GITHUB_HEADERS, json=payload, timeout=60)


## 19) GitHub repo inspection helpers

In [ ]:
def github_view_repo() -> str:
    r = github_request("GET", "git/trees/main?recursive=1")
    if not r.ok:
        return f"GitHub error {r.status_code}: {r.text}"
    tree = r.json().get("tree", [])
    files = [x["path"] for x in tree if x.get("type") == "blob"]
    return "\n".join(files) if files else "Repo is empty"

def github_view_history() -> str:
    r = github_request("GET", "commits?per_page=10")
    if not r.ok:
        return f"History error {r.status_code}: {r.text}"
    commits = r.json()
    lines = [
        f"{c['commit']['author']['name']}: {c['commit']['message']} ({c['commit']['author']['date'][:10]})"
        for c in commits
    ]
    return "\n".join(lines) if lines else "No commits found"

def github_file_exists(path: str) -> bool:
    r = github_request("GET", f"contents/{path}")
    return r.ok


## 20) GitHub file read helper

In [ ]:
def github_read_file(path: str) -> str:
    r = github_request("GET", f"contents/{path}")
    if not r.ok:
        return f"Read file error {r.status_code}: {r.text}"
    content = r.json().get("content")
    if content is None:
        return f"Read file error: no content field returned for {path}"
    return base64.b64decode(content).decode()


## 21) GitHub create / edit helpers

In [ ]:
def github_create_file(path: str, content: str, commit_message: str, agent_name: str) -> str:
    payload = {
        "message": commit_message,
        "content": base64.b64encode(content.encode()).decode(),
        "committer": {"name": agent_name, "email": f"{agent_name.lower()}@starpower.ai"},
    }
    r = github_request("PUT", f"contents/{path}", payload)
    if r.status_code in (200, 201):
        return f"Created {path}"
    return f"GitHub error {r.status_code}: {r.text}"

def github_edit_file(path: str, new_content: str, commit_message: str, agent_name: str) -> str:
    r = github_request("GET", f"contents/{path}")
    if not r.ok:
        return f"Edit file error {r.status_code}: {r.text}"

    sha = r.json()["sha"]
    payload = {
        "message": commit_message,
        "content": base64.b64encode(new_content.encode()).decode(),
        "sha": sha,
        "committer": {"name": agent_name, "email": f"{agent_name.lower()}@starpower.ai"},
    }
    r = github_request("PUT", f"contents/{path}", payload)
    if r.status_code in (200, 201):
        return f"Edited {path}"
    return f"GitHub error {r.status_code}: {r.text}"

def github_upsert_text_file(path: str, content: str, commit_message: str, agent_name: str) -> str:
    if github_file_exists(path):
        return github_edit_file(path, content, commit_message, agent_name)
    return github_create_file(path, content, commit_message, agent_name)


## 22) GitHub folder / rename / delete helpers

In [ ]:
def github_create_folder(path: str, agent_name: str) -> str:
    return github_upsert_text_file(
        f"{path.rstrip('/')}/.gitkeep",
        "",
        f"created folder {path}",
        agent_name,
    )

def github_delete_file(path: str, commit_message: str, agent_name: str) -> str:
    r = github_request("GET", f"contents/{path}")
    if not r.ok:
        return f"Delete error {r.status_code}: {r.text}"

    sha = r.json()["sha"]
    payload = {
        "message": commit_message,
        "sha": sha,
        "committer": {"name": agent_name, "email": f"{agent_name.lower()}@starpower.ai"},
    }
    r = github_request("DELETE", f"contents/{path}", payload)
    if r.status_code == 200:
        return f"Deleted {path}"
    return f"GitHub error {r.status_code}: {r.text}"

def github_rename_file(path: str, new_name_or_path: str, commit_message: str, agent_name: str) -> str:
    content = github_read_file(path.strip())
    if content.startswith("Read file error"):
        return content

    if "/" in new_name_or_path.strip():
        new_path = new_name_or_path.strip()
    else:
        folder = "/".join(path.strip().split("/")[:-1])
        new_path = f"{folder}/{new_name_or_path.strip()}" if folder else new_name_or_path.strip()

    created = github_upsert_text_file(new_path, content, commit_message, agent_name)
    if not created.startswith("Created") and not created.startswith("Edited"):
        return created

    deleted = github_delete_file(path.strip(), f"cleanup after rename -> {path}", agent_name)
    if not deleted.startswith("Deleted"):
        return f"Rename partial success: moved data to {new_path}, but delete failed -> {deleted}"

    return f"Renamed to {new_path}"


## 23) Async GitHub execution wrapper

In [ ]:
async def run_github(fn, *args) -> str:
    log("GITHUB", f"CALL -> {fn.__name__}{args}")
    result = await asyncio.to_thread(fn, *args)
    if PRINT_TOOL_RESULTS:
        log("GITHUB", f"RESULT ->\n{result}")
    return result


## 24) Tavily web-search helper

In [ ]:
async def web_search(query: str) -> str:
    wait = 5

    def _call():
        return tavily.search(query=query, max_results=3)

    while True:
        try:
            log("WEB", f"SEARCH -> {query}")
            r = await asyncio.to_thread(_call)
            result = json.dumps(
                [{"title": x["title"], "snippet": x["content"]} for x in r["results"]],
                ensure_ascii=False,
                indent=2,
            )
            if PRINT_TOOL_RESULTS:
                log("WEB", f"RESULTS ->\n{result}")
            return result
        except Exception as e:
            log("WEB", f"⚠️ Tavily error: {e} — retrying in {wait}s")
            await asyncio.sleep(wait)
            wait = min(wait * 2, 60)


## 25) Telegram send helper

In [ ]:
async def safe_send_message(bot: Bot, chat_id: int, text: str, speaker: str = "BOT") -> bool:
    try:
        log(speaker, f"TELEGRAM SEND -> {text}")
        await bot.send_message(chat_id=chat_id, text=text)
        return True
    except Exception as e:
        log(speaker, f"⚠️ Telegram send error: {e}")
        return False


## 26) Top-level signal definitions

In [ ]:
TOP_SIGNALS = ["{websearch}", "{filemanagement}", "{communicate}"]

CHAT_CONTROL_COMMANDS = [
    "{mutegroupchat}",
    "{unmutegroupchat}",
    "{mutebotchat}",
    "{unmutebotchat}",
]

SEND_COMMAND = "{send}"

ALL_SPECIAL_COMMANDS = TOP_SIGNALS + CHAT_CONTROL_COMMANDS + [SEND_COMMAND]


## 27) File-action labels (plain language, not nested tool syntax)

In [ ]:
FILE_ACTION_LABELS = {
    "view the repo": "view_repo",
    "read a file": "read_file",
    "create a file": "create_file",
    "edit a file": "edit_file",
    "create a folder": "create_folder",
    "rename a file": "rename_file",
    "delete a file": "delete_file",
    "view history": "view_history",
}


## 28) Choice normalization helpers

In [ ]:
def extract_top_signals(text: str) -> List[str]:
    lowered = str(text).lower()
    return [sig for sig in TOP_SIGNALS if sig in lowered]

def extract_commands_in_order(text: str) -> List[str]:
    lowered = str(text).lower()
    hits: List[Tuple[int, str]] = []
    for cmd in ALL_SPECIAL_COMMANDS:
        start = 0
        while True:
            idx = lowered.find(cmd, start)
            if idx == -1:
                break
            hits.append((idx, cmd))
            start = idx + len(cmd)
    hits.sort(key=lambda x: x[0])
    return [cmd for _, cmd in hits]

def contains_any_signal(text: str) -> bool:
    lowered = str(text).lower()
    return any(cmd in lowered for cmd in TOP_SIGNALS)

def contains_special_command(text: str) -> bool:
    lowered = str(text).lower()
    return any(cmd in lowered for cmd in ALL_SPECIAL_COMMANDS)

def normalize_choice(text: str) -> str:
    return normalize_ws(str(text)).strip().lower()

def resolve_chat_choice(text: str) -> Optional[str]:
    t = normalize_choice(text)
    if t in {"groupchat", "group chat", "read the groupchat", "read groupchat"}:
        return "groupchat"
    if t in {"botchat", "bot chat", "read the botchat", "read the bot chat", "read botchat", "read bot chat"}:
        return "botchat"
    return None

def resolve_file_action(text: str) -> Optional[str]:
    t = normalize_choice(text)
    for label, value in FILE_ACTION_LABELS.items():
        if t == label or t == value.replace("_", " "):
            return value
    return None

def split_first_command_and_rest(text: str, command: str) -> Tuple[str, str]:
    lowered = str(text).lower()
    idx = lowered.find(command)
    if idx == -1:
        return "", str(text)
    raw = str(text)
    end = idx + len(command)
    return raw[idx:end], raw[end:].strip()

def parse_prefill_steps(rest: str) -> List[str]:
    if not rest.strip():
        return []

    lines = [line.strip() for line in rest.splitlines() if line.strip()]
    if not lines:
        return []

    if len(lines) == 1:
        return [lines[0]]

    merged: List[str] = []
    message_started = False
    message_parts: List[str] = []

    for line in lines:
        if message_started:
            message_parts.append(line)
            continue

        if line.startswith("- "):
            merged.append(line[2:].strip())
            continue

        if len(merged) < 2:
            merged.append(line)
            continue

        message_started = True
        message_parts.append(line)

    if message_parts:
        merged.append("\n".join(message_parts).strip())

    return [item for item in merged if item]

def is_yes(text: str) -> bool:
    return normalize_choice(text) == "yes"

def is_edit_response(text: str) -> bool:
    return normalize_choice(text) == "edit response"


## 29) Single-step validation helpers

In [ ]:
def answer_has_multiple_tool_steps(text: str) -> bool:
    return len(extract_top_signals(text)) > 1

def invalid_single_step_reply(text: str) -> bool:
    return answer_has_multiple_tool_steps(text)

def invalid_user_visible_message(text: str) -> bool:
    return contains_special_command(text)

async def get_step_input(agent, prompt: str, phase: str) -> Optional[str]:
    if getattr(agent, "prefill_steps", None):
        candidate = agent.prefill_steps.pop(0)
        confirm = await agent.ask(
            "This is the next step you already selected:\n"
            f"{candidate}\n\n"
            "Is this what you want for this step?\n"
            "Reply with one option only:\n"
            "- Yes\n"
            "- Edit Response",
            store_prompt=False,
            store_response=True,
            phase=f"{phase} confirm",
        )
        if is_yes(confirm):
            return candidate
        if not is_edit_response(confirm):
            agent.system_notice("Reply with one option only: Yes or Edit Response.")
            return None

    return await agent.ask(
        prompt,
        store_prompt=False,
        store_response=True,
        phase=phase,
    )


## 30) Pending-response task structure

In [ ]:
@dataclass
class PendingResponseTask:
    chat_type: str
    created_at: str
    thread_snapshot: str
    reason: str = "manual send flow"


## 31) Streaming GLM helper

In [ ]:
async def safe_glm(client: ZhipuAI, messages: List[Dict[str, str]], agent_name: str = "system") -> str:
    wait = 5

    def _nonstream_call() -> str:
        kwargs = {
            "model": MODEL_NAME,
            "messages": messages,
            "max_tokens": MODEL_MAX_TOKENS,
            "temperature": MODEL_TEMPERATURE,
        }
        if ENABLE_REASONING_MODE:
            kwargs["thinking"] = {"type": "enabled"}
        res = client.chat.completions.create(**kwargs)
        return res.choices[0].message.content

    def _stream_call() -> str:
        kwargs = {
            "model": MODEL_NAME,
            "messages": messages,
            "max_tokens": MODEL_MAX_TOKENS,
            "temperature": MODEL_TEMPERATURE,
            "stream": True,
        }
        if ENABLE_REASONING_MODE:
            kwargs["thinking"] = {"type": "enabled"}

        stream = client.chat.completions.create(**kwargs)

        content_parts: List[str] = []
        reasoning_parts: List[str] = []

        if PRINT_STREAM_HEADERS:
            print(f"[{ts()}] [{agent_name}] STREAM START", flush=True)

        for chunk in stream:
            try:
                choice = chunk.choices[0]
                delta = getattr(choice, "delta", None)
                if delta is None:
                    continue

                reasoning = getattr(delta, "reasoning_content", None)
                content = getattr(delta, "content", None)

                if reasoning:
                    reasoning_parts.append(str(reasoning))
                    print(str(reasoning), end="", flush=True)

                if content:
                    content_parts.append(str(content))
                    print(str(content), end="", flush=True)
            except Exception as inner_e:
                print(f"\n[{ts()}] [{agent_name}] STREAM CHUNK ERROR -> {inner_e}", flush=True)

        print("", flush=True)

        merged = "".join(content_parts).strip()
        if merged:
            print(f"[{ts()}] [{agent_name}] STREAM DONE | chars={len(merged)}", flush=True)
            return merged

        fallback = "".join(reasoning_parts).strip()
        print(f"[{ts()}] [{agent_name}] STREAM DONE | chars={len(fallback)}", flush=True)
        return fallback

    while True:
        try:
            log(agent_name, f"GLM REQUEST START | messages={len(messages)}")
            if ENABLE_STREAMING:
                try:
                    return await asyncio.to_thread(_stream_call)
                except Exception as stream_e:
                    log(agent_name, f"STREAM fallback -> {stream_e}")
            result = await asyncio.to_thread(_nonstream_call)
            log(agent_name, f"GLM REQUEST DONE | chars={len(result)}")
            return result
        except Exception as e:
            log(agent_name, f"⚠️ GLM error: {e} — retrying in {wait}s")
            await asyncio.sleep(wait)
            wait = min(wait * 2, 120)


## 32) Agent class — construction and dashboard state

In [ ]:
class Agent:
    def __init__(self, name: str, token: str, persona_prompt: str, teammate_name: str, zhipu_client: ZhipuAI, rate_limit: int):
        self.name = name
        self.bot = Bot(token=token)
        self.persona_prompt = persona_prompt
        self.teammate_name = teammate_name
        self.zhipu = zhipu_client
        self.rate_limit = rate_limit
        self.peers: List["Agent"] = []

        self.glm_lock = asyncio.Lock()
        self.last_glm_time = 0.0

        self.memory_events: List[Dict[str, str]] = []
        self.group_thread: List[Dict[str, str]] = []
        self.bot_thread: List[Dict[str, str]] = []

        self.unread_group: List[Dict[str, str]] = []
        self.unread_bot: List[Dict[str, str]] = []

        self.group_chat_muted: bool = True
        self.bot_chat_muted: bool = True

        self.system_notifications: List[str] = []
        self.personal_prompt: str = ""
        self.last_signal: str = "—"
        self.last_tool: str = "—"
        self.last_phase: str = "booting"
        self.last_thought: str = ""
        self.group_chat_id: Optional[int] = None
        self.prefill_steps: List[str] = []
        self.context_trim_count: int = 0
        self.initiation_used: bool = False

        self.wake_event = asyncio.Event()

        self.refresh_dashboard()

    def refresh_dashboard(self) -> None:
        ui.set_card(self.name, {
            "phase": self.last_phase,
            "group_state": f"{'muted' if self.group_chat_muted else 'unmuted'} | unread {len(self.unread_group)}",
            "bot_state": f"{'muted' if self.bot_chat_muted else 'unmuted'} | unread {len(self.unread_bot)}",
            "last_signal": self.last_signal,
            "last_tool": self.last_tool,
            "current_status": clip_text(self.current_system_status_text(), MAX_GOAL_CHARS),
            "personal_prompt": clip_text(self.personal_prompt, MAX_GOAL_CHARS),
            "last_thought": clip_text(self.last_thought, 2200),
        })


## 33) Agent class — notifications and memory events

In [ ]:
def _agent_memory_methods():
    def remember(self, role: str, content: str, kind: str = "general", dedupe: bool = True) -> bool:
        content = str(content)
        if dedupe and self.memory_events:
            prev = self.memory_events[-1]
            if prev["role"] == role and normalize_ws(prev["content"]) == normalize_ws(content):
                log(self.name, f"↺ duplicate {kind} suppressed from context memory")
                return False

        self.memory_events.append({
            "role": role,
            "content": content,
            "kind": kind,
            "time": ts(),
        })
        return True

    def inject_memory(self, content: str, label: str = "MEMORY", dedupe: bool = True) -> None:
        self.remember("user", content, kind=label, dedupe=dedupe)
        log(self.name, f"{label} INJECT ->\n{content}")

    def add_system_notification(self, message: str, wake: bool = True) -> None:
        self.system_notifications.append(message)
        log(self.name, f"🔔 notification -> {message}")
        if wake:
            self.wake_event.set()
        self.refresh_dashboard()

    def system_notice(self, content: str, wake: bool = True) -> None:
        text = f"System notice: {content}"
        self.add_system_notification(text, wake=wake)
        self.inject_memory(text, label="SYSTEM", dedupe=False)

    def system_info(self, content: str, wake: bool = True) -> None:
        text = f"System info: {content}"
        self.add_system_notification(text, wake=wake)
        self.inject_memory(text, label="SYSTEM", dedupe=False)

    def clear_ephemeral_system_notifications(self) -> None:
        self.system_notifications = []

    def current_unmuted_chat(self) -> Optional[str]:
        if not self.group_chat_muted:
            return "groupchat"
        if not self.bot_chat_muted:
            return "botchat"
        return None

    def set_chat_state(self, chat_type: str, muted: bool) -> None:
        if chat_type == "groupchat":
            self.group_chat_muted = muted
            if not muted:
                self.bot_chat_muted = True
        elif chat_type == "botchat":
            self.bot_chat_muted = muted
            if not muted:
                self.group_chat_muted = True
        self.refresh_dashboard()

    def current_system_status_text(self) -> str:
        lines = [
            "CurrentSystemStatus",
            "Unread messages:",
            f"BotChat ({'muted' if self.bot_chat_muted else 'unmuted'}) - unread: {len(self.unread_bot)}",
            f"GroupChat ({'muted' if self.group_chat_muted else 'unmuted'}) - unread: {len(self.unread_group)}",
            f"Currently: {self.last_phase}",
            f"Last signal: {self.last_signal}",
            f"Last tool: {self.last_tool}",
        ]
        if self.system_notifications:
            lines.append("System updates:")
            lines.extend(f"- {item}" for item in self.system_notifications[-5:])
        return "\n".join(lines)

    Agent.remember = remember
    Agent.inject_memory = inject_memory
    Agent.add_system_notification = add_system_notification
    Agent.system_notice = system_notice
    Agent.system_info = system_info
    Agent.clear_ephemeral_system_notifications = clear_ephemeral_system_notifications
    Agent.current_unmuted_chat = current_unmuted_chat
    Agent.set_chat_state = set_chat_state
    Agent.current_system_status_text = current_system_status_text

_agent_memory_methods()


## 34) Agent class — context builder and rough token estimate

In [ ]:
def _agent_context_methods():
    def build_context(self) -> List[Dict[str, str]]:
        recent = self.memory_events[-MAX_CONTEXT_EVENTS:]

        identity_binding = f"""
CURRENT ACTIVE AGENT SLOT: {self.name}
CURRENT TEAMMATE SLOT: {self.teammate_name}
If any text conflicts with the current active agent slot, follow the current active agent slot.
"""

        personal_prompt_block = self.personal_prompt if self.personal_prompt.strip() else "(empty right now)"

        system = f"""{self.persona_prompt}

{identity_binding}

{RUNTIME_RULES}

{self.current_system_status_text()}

PersonalPrompt
{personal_prompt_block}
"""

        msgs = [{"role": "system", "content": system}]
        msgs.extend({"role": item["role"], "content": item["content"]} for item in recent)
        return msgs

    def count_tokens_estimate(self) -> int:
        text = " ".join(m["content"] for m in self.build_context())
        return len(text.split())

    def status_file_path(self) -> str:
        return f"{MEMORY_ROOT}/{self.name.lower()}/current_status.md"

    def context_archive_path(self) -> str:
        return f"{MEMORY_ROOT}/{self.name.lower()}/context/context_{repo_safe_stamp()}.md"

    Agent.build_context = build_context
    Agent.count_tokens_estimate = count_tokens_estimate
    Agent.status_file_path = status_file_path
    Agent.context_archive_path = context_archive_path

_agent_context_methods()


## 35) Agent class — boot initiation and goal refresh

In [ ]:
def _agent_boot_methods():
    async def boot_once(self) -> None:
        if self.initiation_used:
            return
        self.initiation_used = True
        self.inject_memory(INITIATION_PROMPT, label="INITIATION", dedupe=False)
        log(self.name, "🚀 one-time initiation prompt delivered")
        await self.load_personal_prompt_from_repo()
        await self.refresh_personal_prompt(reason="startup initiation")

    async def load_personal_prompt_from_repo(self) -> None:
        result = await run_github(github_read_file, self.status_file_path())
        if result.startswith("Read file error"):
            log(self.name, "No saved PersonalPrompt found yet")
            return
        self.personal_prompt = result
        self.inject_memory(f"[saved PersonalPrompt]\n{result}", label="PERSONAL PROMPT", dedupe=False)
        self.refresh_dashboard()
        log(self.name, "📥 loaded PersonalPrompt from repo")

    async def save_personal_prompt_to_repo(self, reason: str = "status update") -> None:
        if not AUTO_SAVE_STATUS or not self.personal_prompt.strip():
            return
        result = await run_github(
            github_upsert_text_file,
            self.status_file_path(),
            self.personal_prompt,
            f"{self.name} personal prompt update ({reason})",
            self.name,
        )
        log(self.name, f"💾 PersonalPrompt save -> {result}")

    async def refresh_personal_prompt(self, reason: str = "runtime") -> None:
        prompt = (
            "Update your PersonalPrompt for yourself.\n"
            "Reply in this exact plain format:\n\n"
            "PersonalPrompt\n"
            "Current goal:\n"
            "<your current goal>\n\n"
            "Progress:\n"
            "<your current progress>\n\n"
            "Checklist:\n"
            "- [ ] item\n\n"
            "Need to remember:\n"
            "<what matters right now>\n\n"
            "Keep it honest, compact, and useful.\n"
            f"Reason: {reason}"
        )
        result = await self.ask(
            prompt,
            store_prompt=False,
            store_response=False,
            phase="personal prompt update",
        )
        self.personal_prompt = result
        self.last_tool = "personal prompt"
        self.refresh_dashboard()
        await self.save_personal_prompt_to_repo(reason=reason)
        log(self.name, "📝 PersonalPrompt refreshed")

    async def archive_and_trim_context(self, reason: str = "context trim") -> None:
        self.last_phase = "context trim"
        self.refresh_dashboard()
        await self.refresh_personal_prompt(reason=reason)

        snapshot = (
            f"{self.current_system_status_text()}\n\n"
            f"PersonalPrompt\n{self.personal_prompt}\n\n"
            f"Context snapshot\n{render_memory_events(self.memory_events)}"
        )
        result = await run_github(
            github_upsert_text_file,
            self.context_archive_path(),
            snapshot,
            f"{self.name} context trim snapshot",
            self.name,
        )
        self.context_trim_count += 1
        self.memory_events = self.memory_events[-TRIM_KEEP_EVENTS:]
        self.inject_memory(
            f"[context trim]\n{result}\nYour PersonalPrompt is saved. Continue from what matters now.",
            label="SYSTEM",
            dedupe=False,
        )
        self.refresh_dashboard()
        log(self.name, f"✂️ context trimmed ({self.context_trim_count})")

    Agent.boot_once = boot_once
    Agent.load_personal_prompt_from_repo = load_personal_prompt_from_repo
    Agent.save_personal_prompt_to_repo = save_personal_prompt_to_repo
    Agent.refresh_personal_prompt = refresh_personal_prompt
    Agent.archive_and_trim_context = archive_and_trim_context

_agent_boot_methods()


## 36) Agent class — model call wrapper

In [ ]:
def _agent_ask_method():
    async def ask(self, prompt: str, store_prompt: bool = False, store_response: bool = True, phase: str = "thinking") -> str:
        async with self.glm_lock:
            gap = time.time() - self.last_glm_time
            if gap < self.rate_limit:
                wait = self.rate_limit - gap
                self.last_phase = f"cooldown {wait:.1f}s"
                self.refresh_dashboard()
                log(self.name, f"⏳ cooldown — waiting {wait:.1f}s")
                await asyncio.sleep(wait)

            self.last_phase = phase
            self.refresh_dashboard()

            msgs = self.build_context()
            msgs.append({"role": "user", "content": prompt})

            if PRINT_FULL_SYSTEM_CONTEXT:
                log(self.name, f"SYSTEM CONTEXT ->\n{msgs[0]['content']}")
            log(self.name, f"PROMPT -> {prompt}")

            if store_prompt:
                self.remember("user", prompt, kind="PROMPT", dedupe=False)

            response = await safe_glm(self.zhipu, msgs, self.name)
            self.last_glm_time = time.time()

            if store_response:
                self.remember("assistant", response, kind="THOUGHT", dedupe=True)

            self.last_thought = response
            self.last_phase = "thought complete"
            self.refresh_dashboard()

            log(self.name, f"THOUGHT ->\n{response}")
            return response

    Agent.ask = ask

_agent_ask_method()


## 37) Agent class — free-thinking loop

In [ ]:
def _agent_loop_method():
    async def think_loop(self) -> None:
        self.last_phase = "startup"
        self.refresh_dashboard()

        await self.boot_once()

        while True:
            try:
                rough_tokens = self.count_tokens_estimate()

                if rough_tokens > ROUGH_CONTEXT_TOKEN_TRIM:
                    self.system_info(
                        "Your context is getting heavy. Update your PersonalPrompt and save what matters now.",
                        wake=False,
                    )
                    await self.archive_and_trim_context(reason="context getting heavy")

                elif rough_tokens > ROUGH_CONTEXT_TOKEN_WARNING:
                    self.system_info(
                        "Your context is getting heavy. Keep your next steps concrete and refresh your PersonalPrompt soon.",
                        wake=False,
                    )

                free_prompt = (
                    "Continue your autonomous thinking.\n"
                    "Choose one useful next step.\n"
                    "If you need a tool, use exactly one top-level signal.\n"
                    "If you want instant messages, use one chat control command or {send}.\n"
                    "If you already know the next step inside a tool, you may put that step on the next line after the signal and the system will walk you through it one step at a time."
                )

                thought = await self.ask(
                    free_prompt,
                    store_prompt=False,
                    store_response=True,
                    phase="thinking",
                )

                commands = extract_commands_in_order(thought)

                if len(commands) > 1:
                    self.system_notice(
                        "You used more than one signal or chat control command in one reply. Use one at a time."
                    )
                    self.prefill_steps = []
                    self.clear_ephemeral_system_notifications()
                    try:
                        await asyncio.wait_for(self.wake_event.wait(), timeout=self.rate_limit)
                        self.wake_event.clear()
                    except asyncio.TimeoutError:
                        pass
                    continue

                if len(commands) == 1:
                    command = commands[0]
                    self.last_signal = command
                    self.refresh_dashboard()

                    _, rest = split_first_command_and_rest(thought, command)
                    self.prefill_steps = parse_prefill_steps(rest)

                    if command in CHAT_CONTROL_COMMANDS:
                        await chat_control_flow(self, command)
                    elif command == SEND_COMMAND:
                        await send_flow(self)
                    elif command in TOP_SIGNALS:
                        await tool_assist(self, command)
                    else:
                        self.system_notice(f"Unknown signal: {command}")

                    self.prefill_steps = []

                self.clear_ephemeral_system_notifications()

                try:
                    await asyncio.wait_for(self.wake_event.wait(), timeout=self.rate_limit)
                    self.wake_event.clear()
                except asyncio.TimeoutError:
                    pass

            except Exception as e:
                self.last_phase = "loop error"
                self.refresh_dashboard()
                log(self.name, f"⚠️ think_loop error: {e}\n{traceback.format_exc()}")
                await asyncio.sleep(10)

    Agent.think_loop = think_loop

_agent_loop_method()


## 38) Communication tool flow

In [ ]:
async def chat_control_flow(agent: Agent, command: str) -> None:
    agent.last_tool = command
    agent.refresh_dashboard()

    if command == "{mutegroupchat}":
        agent.set_chat_state("groupchat", True)
        agent.system_info("GroupChat is now muted.")
        return

    if command == "{unmutegroupchat}":
        agent.set_chat_state("groupchat", False)
        agent.system_info("GroupChat is now unmuted. BotChat is muted.")
        return

    if command == "{mutebotchat}":
        agent.set_chat_state("botchat", True)
        agent.system_info("BotChat is now muted.")
        return

    if command == "{unmutebotchat}":
        agent.set_chat_state("botchat", False)
        agent.system_info("BotChat is now unmuted. GroupChat is muted.")
        return

    agent.system_notice("That chat control command is not valid.")

async def send_flow(agent: Agent, forced_chat: Optional[str] = None) -> None:
    agent.last_tool = "send"
    agent.refresh_dashboard()

    target_chat = forced_chat or agent.current_unmuted_chat()
    if target_chat is None:
        agent.system_notice("No chat is unmuted. Use {unmuteGroupChat} or {unmuteBotChat} first, or use {communicate}.")
        return

    message = await get_step_input(
        agent,
        "Please write the message you would like to send.",
        phase="send message",
    )
    if message is None:
        return

    if invalid_user_visible_message(message):
        agent.system_notice("Messages must not contain signals or chat control commands.")
        return

    while True:
        confirm = await agent.ask(
            "Is this the message you would like to send?\n\n"
            f"{message}\n\n"
            "Reply with one option only:\n"
            "- Yes\n"
            "- Edit Response",
            store_prompt=False,
            store_response=True,
            phase="send confirm",
        )

        if is_yes(confirm):
            break

        if is_edit_response(confirm):
            message = await agent.ask(
                "Please write the message you would like to send.",
                store_prompt=False,
                store_response=True,
                phase="send message edit",
            )
            if invalid_user_visible_message(message):
                agent.system_notice("Messages must not contain signals or chat control commands.")
                return
            continue

        agent.system_notice("Reply with one option only: Yes or Edit Response.")
        return

    if target_chat == "groupchat":
        if agent.group_chat_id is None:
            agent.system_notice("GroupChat is not locked yet. Wait until a Telegram group message arrives first.")
            return

        ok = await safe_send_message(agent.bot, agent.group_chat_id, message, speaker=agent.name)
        if not ok:
            agent.system_notice("GroupChat send failed.")
            return

        entry = make_entry(agent.name, message)
        for peer in [agent] + agent.peers:
            peer.group_thread.append(entry)
            if peer is not agent:
                peer.unread_group.append(entry)
                if not peer.group_chat_muted:
                    peer.inject_memory(
                        f"[most recent GroupChat message]\n[{entry['time']}] {entry['speaker']}: {entry['content']}",
                        label="GROUPCHAT",
                        dedupe=False,
                    )
                    peer.add_system_notification(
                        f"Most recent GroupChat message from {entry['speaker']}: {clip_text(entry['content'], 180)}",
                        wake=True,
                    )
        log(agent.name, f"✉️ GroupChat -> {message}")

    else:
        entry = make_entry(agent.name, message)
        agent.bot_thread.append(entry)
        for peer in agent.peers:
            peer.bot_thread.append(entry)
            peer.unread_bot.append(entry)
            if not peer.bot_chat_muted:
                peer.inject_memory(
                    f"[most recent BotChat message]\n[{entry['time']}] {entry['speaker']}: {entry['content']}",
                    label="BOTCHAT",
                    dedupe=False,
                )
                peer.add_system_notification(
                    f"Most recent BotChat message from {entry['speaker']}: {clip_text(entry['content'], 180)}",
                    wake=True,
                )
        log(agent.name, f"🤖 BotChat -> {message}")

    agent.refresh_dashboard()

async def communicate_flow(agent: Agent) -> None:
    agent.last_tool = "communicate"
    agent.refresh_dashboard()

    choice = await get_step_input(
        agent,
        "Please choose your option:\n- GroupChat\n- BotChat",
        phase="communicate choice",
    )
    if choice is None:
        return

    if invalid_single_step_reply(choice):
        agent.system_notice("Reply with only one chat choice.")
        return

    chosen_chat = resolve_chat_choice(choice)
    if chosen_chat is None:
        agent.system_notice("That was not a valid chat choice. Reply with only GroupChat or BotChat.")
        return

    if chosen_chat == "groupchat":
        thread_text = render_entries(agent.group_thread)
        unread_text = render_entries(agent.unread_group)
        agent.inject_memory(f"[GroupChat thread]\n{thread_text}", label="GROUP READ", dedupe=False)
        agent.inject_memory(f"[GroupChat unread since last read]\n{unread_text}", label="GROUP READ", dedupe=False)
        agent.unread_group = []
    else:
        thread_text = render_entries(agent.bot_thread)
        unread_text = render_entries(agent.unread_bot)
        agent.inject_memory(f"[BotChat thread]\n{thread_text}", label="BOT READ", dedupe=False)
        agent.inject_memory(f"[BotChat unread since last read]\n{unread_text}", label="BOT READ", dedupe=False)
        agent.unread_bot = []

    agent.refresh_dashboard()

    next_step = await get_step_input(
        agent,
        "What would you like to do next?\nReply with one option only:\n- {send}\n- think\n- no response",
        phase="communicate next step",
    )
    if next_step is None:
        return

    choice_norm = normalize_choice(next_step)
    if choice_norm == "{send}" or choice_norm == "send":
        await send_flow(agent, forced_chat=chosen_chat)
        return

    if choice_norm == "think":
        agent.system_info(f"You read the {chosen_chat} and chose to think.")
        return

    if choice_norm == "no response":
        agent.system_info(f"You read the {chosen_chat} and chose no response.")
        return

    agent.system_notice("Reply with one option only: {send}, think, or no response.")


## 39) Web-search tool flow

In [ ]:
async def websearch_flow(agent: Agent) -> None:
    agent.last_tool = "websearch"
    agent.refresh_dashboard()

    query = await get_step_input(
        agent,
        "What would you like to search for? Please reply with only the search query.",
        phase="websearch query",
    )
    if query is None:
        return

    if invalid_single_step_reply(query):
        agent.system_notice("Reply with only the search query.")
        return

    if contains_any_signal(query):
        agent.system_notice("Do not include another tool signal while answering the web search step.")
        return

    result = await web_search(query)
    agent.inject_memory(f"[web search results for '{query}']\n{result}", label="WEB RESULT", dedupe=False)


## 40) File-management tool flow — choose action

In [ ]:
async def filemanagement_flow(agent: Agent) -> None:
    agent.last_tool = "filemanagement"
    agent.refresh_dashboard()

    choice = await get_step_input(
        agent,
        "Please choose your option:\n"
        "- view the repo\n"
        "- read a file\n"
        "- create a file\n"
        "- edit a file\n"
        "- create a folder\n"
        "- rename a file\n"
        "- delete a file\n"
        "- view history",
        phase="file action choice",
    )
    if choice is None:
        return

    if invalid_single_step_reply(choice):
        agent.system_notice("Reply with only one file action.")
        return

    if contains_any_signal(choice):
        agent.system_notice("Do not include another tool signal while answering the file action step.")
        return

    action = resolve_file_action(choice)
    if action is None:
        agent.system_notice("That was not a valid file action. Reply with one listed option only.")
        return

    agent.last_tool = action
    agent.refresh_dashboard()

    if action == "view_repo":
        result = await run_github(github_view_repo)
        agent.inject_memory(f"[repo file tree]\n{result}", label="FILE", dedupe=False)
        return

    if action == "view_history":
        result = await run_github(github_view_history)
        agent.inject_memory(f"[recent commits]\n{result}", label="FILE", dedupe=False)
        return

    if action == "read_file":
        await file_read_flow(agent)
        return

    if action == "create_file":
        await file_create_flow(agent)
        return

    if action == "edit_file":
        await file_edit_flow(agent)
        return

    if action == "create_folder":
        await file_create_folder_flow(agent)
        return

    if action == "rename_file":
        await file_rename_flow(agent)
        return

    if action == "delete_file":
        await file_delete_flow(agent)
        return


## 41) File-management flow — read file

In [ ]:
async def file_read_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "Which file would you like to read? Please reply with only the path.",
        phase="read file path",
    )
    if path is None:
        return

    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the file path for the read file step.")
        return

    result = await run_github(github_read_file, path.strip())
    agent.inject_memory(f"[contents of {path.strip()}]\n{result}", label="FILE", dedupe=False)


## 42) File-management flow — create file

In [ ]:
async def file_create_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "What is the full path for the new file? Please reply with only the path.",
        phase="create file path",
    )
    if path is None:
        return
    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the new file path.")
        return

    content = await get_step_input(
        agent,
        "Please reply with the full content for the new file.",
        phase="create file content",
    )
    if content is None:
        return
    if invalid_single_step_reply(content):
        agent.system_notice("Do not combine multiple tool steps while providing file content.")
        return

    commit = await get_step_input(
        agent,
        "Please reply with the commit message.",
        phase="create file commit",
    )
    if commit is None:
        return
    if invalid_single_step_reply(commit) or contains_any_signal(commit):
        agent.system_notice("Reply with only the commit message.")
        return

    result = await run_github(github_create_file, path.strip(), content, commit, agent.name)
    agent.inject_memory(f"[file created]\n{result}", label="FILE", dedupe=False)


## 43) File-management flow — edit file

In [ ]:
async def file_edit_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "Which file would you like to edit? Please reply with only the path.",
        phase="edit file path",
    )
    if path is None:
        return
    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the file path for the edit file step.")
        return

    current = await run_github(github_read_file, path.strip())
    agent.inject_memory(f"[current content of {path.strip()}]\n{current}", label="FILE", dedupe=False)

    new_content = await get_step_input(
        agent,
        "Please reply with the full new content for this file.",
        phase="edit file content",
    )
    if new_content is None:
        return
    if invalid_single_step_reply(new_content):
        agent.system_notice("Do not combine multiple tool steps while providing edited file content.")
        return

    commit = await get_step_input(
        agent,
        "Please reply with the commit message.",
        phase="edit file commit",
    )
    if commit is None:
        return
    if invalid_single_step_reply(commit) or contains_any_signal(commit):
        agent.system_notice("Reply with only the commit message.")
        return

    result = await run_github(github_edit_file, path.strip(), new_content, commit, agent.name)
    agent.inject_memory(f"[file edited]\n{result}", label="FILE", dedupe=False)


## 44) File-management flow — create folder

In [ ]:
async def file_create_folder_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "What is the folder path you want to create? Please reply with only the path.",
        phase="create folder path",
    )
    if path is None:
        return
    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the folder path.")
        return

    result = await run_github(github_create_folder, path.strip(), agent.name)
    agent.inject_memory(f"[folder created]\n{result}", label="FILE", dedupe=False)


## 45) File-management flow — rename file

In [ ]:
async def file_rename_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "Which file would you like to rename? Please reply with only the current path.",
        phase="rename file current path",
    )
    if path is None:
        return
    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the current file path.")
        return

    new_name_or_path = await get_step_input(
        agent,
        "Please reply with the new filename or the full new path.",
        phase="rename file new path",
    )
    if new_name_or_path is None:
        return
    if invalid_single_step_reply(new_name_or_path) or contains_any_signal(new_name_or_path):
        agent.system_notice("Reply with only the new filename or full new path.")
        return

    commit = await get_step_input(
        agent,
        "Please reply with the commit message.",
        phase="rename file commit",
    )
    if commit is None:
        return
    if invalid_single_step_reply(commit) or contains_any_signal(commit):
        agent.system_notice("Reply with only the commit message.")
        return

    result = await run_github(github_rename_file, path.strip(), new_name_or_path.strip(), commit, agent.name)
    agent.inject_memory(f"[file renamed]\n{result}", label="FILE", dedupe=False)


## 46) File-management flow — delete file

In [ ]:
async def file_delete_flow(agent: Agent) -> None:
    path = await get_step_input(
        agent,
        "Which file would you like to delete? Please reply with only the path.",
        phase="delete file path",
    )
    if path is None:
        return
    if invalid_single_step_reply(path) or contains_any_signal(path):
        agent.system_notice("Reply with only the file path to delete.")
        return

    confirm = await get_step_input(
        agent,
        "If you are sure, reply exactly DELETE_THIS_FILE.\nIf you do not want to delete it, reply exactly CANCEL_DELETE.",
        phase="delete file confirm",
    )
    if confirm is None:
        return

    confirm_norm = normalize_ws(confirm).strip()
    if confirm_norm not in {"DELETE_THIS_FILE", "CANCEL_DELETE"}:
        agent.system_notice("Reply exactly DELETE_THIS_FILE or CANCEL_DELETE.")
        return

    if confirm_norm == "CANCEL_DELETE":
        agent.system_info("Deletion cancelled.")
        return

    commit = await get_step_input(
        agent,
        "Please reply with the commit message.",
        phase="delete file commit",
    )
    if commit is None:
        return
    if invalid_single_step_reply(commit) or contains_any_signal(commit):
        agent.system_notice("Reply with only the commit message.")
        return

    result = await run_github(github_delete_file, path.strip(), commit, agent.name)
    agent.inject_memory(f"[file deleted]\n{result}", label="FILE", dedupe=False)


## 47) Top-level tool dispatcher

In [ ]:
async def tool_assist(agent: Agent, signal: str) -> None:
    if signal == "{communicate}":
        await communicate_flow(agent)
        return

    if signal == "{websearch}":
        await websearch_flow(agent)
        return

    if signal == "{filemanagement}":
        await filemanagement_flow(agent)
        return

    agent.system_notice(f"Unknown top-level signal: {signal}")


## 48) Telegram long-poll message listener

In [ ]:
async def message_listener(agents: List[Agent]) -> None:
    watcher_bot = Bot(token=WVY_TOK)
    offset = 0
    log("LISTENER", "📡 Listener live — waiting for messages...")

    while True:
        try:
            updates = await watcher_bot.get_updates(
                offset=offset,
                timeout=60,
                allowed_updates=["message", "edited_message"],
            )

            for u in updates:
                offset = u.update_id + 1
                msg = u.message or u.edited_message
                if not msg or not msg.text:
                    continue

                chat_id = getattr(msg, "chat_id", None) or msg.chat.id

                for a in agents:
                    a.group_chat_id = chat_id

                if msg.from_user and not msg.from_user.is_bot:
                    handle = (
                        f"@{msg.from_user.username}"
                        if getattr(msg.from_user, "username", None)
                        else msg.from_user.first_name
                    )
                    txt = msg.text
                    entry = make_entry(handle, txt)

                    log("GROUP", f"USER MESSAGE -> {handle}: {txt}")

                    for a in agents:
                        a.group_thread.append(entry)
                        a.unread_group.append(entry)
                        if not a.group_chat_muted:
                            a.inject_memory(
                                f"[most recent GroupChat message]\n[{entry['time']}] {handle}: {txt}",
                                label="GROUPCHAT",
                                dedupe=False,
                            )
                            a.add_system_notification(
                                f"Most recent GroupChat message from {handle}: {clip_text(txt, 180)}",
                                wake=True,
                            )
                        else:
                            a.refresh_dashboard()
                else:
                    bot_name = msg.from_user.first_name if msg.from_user else "bot"
                    log("GROUP", f"BOT MESSAGE SEEN -> {bot_name}: {msg.text}")

        except Exception as e:
            log("LISTENER", f"⚠️ Listener error: {e}")
            await asyncio.sleep(5)


## 49) Boot banner

In [ ]:
def print_boot_banner() -> None:
    print("=" * 92, flush=True)
    print("STARPOWER PRODUCTION NOTEBOOK // WVY + SAVVY", flush=True)
    print("=" * 92, flush=True)
    print("Both agents start active immediately.", flush=True)
    print("Rate limit = seconds between each agent model call.", flush=True)
    print("Recent turns to keep = working context turns kept in memory, not the full model context window.", flush=True)
    print("=" * 92, flush=True)


## 50) Main runtime assembly

In [ ]:
async def main() -> None:
    print_boot_banner()
    show_section(
        "LAUNCH SEQUENCE",
        "Starting the listener, both agents, the live dashboard, and the autonomy loop without waiting for a new group message."
    )

    wvy = Agent(
        name="WVY",
        token=WVY_TOK,
        persona_prompt=WVY_PROMPT,
        teammate_name="Savvy",
        zhipu_client=WVY_ZAI,
        rate_limit=WVY_RATE_LIMIT,
    )

    savvy = Agent(
        name="Savvy",
        token=SAVVY_TOK,
        persona_prompt=SAVVY_PROMPT,
        teammate_name="WVY",
        zhipu_client=SAVVY_ZAI,
        rate_limit=SAVVY_RATE_LIMIT,
    )

    wvy.peers = [savvy]
    savvy.peers = [wvy]

    log("BOOT", "🔥 Booting WVY & Savvy... both minds start active now.")

    await asyncio.gather(
        message_listener([wvy, savvy]),
        wvy.think_loop(),
        savvy.think_loop(),
    )


## 51) Final run cell

In [ ]:
asyncio.run(main())

## 52) Notes for live debugging

- `WVY_RATE_LIMIT` and `SAVVY_RATE_LIMIT` are the cooldown in seconds between each agent model call.
- `MAX_CONTEXT_EVENTS` is the number of recent memory events kept in the working context.
- `ROUGH_CONTEXT_TOKEN_WARNING` is the warning line.
- `ROUGH_CONTEXT_TOKEN_TRIM` is the save-and-trim line.
- `memory/<agent>/current_status.md` stores the PersonalPrompt.
- `memory/<agent>/context/` stores saved context snapshots before trimming.
- GroupChat and BotChat can be muted or unmuted, but only one chat can be unmuted at a time.
- When a chat is unmuted, the most recent message is injected into thoughts and `{send}` is the message gate.
